# Probability and Statistics Foundations Lab

## Distributions, conditioning, covariance, Monte Carlo, inference, and Bayesian updating

This guided lab accompanies the [probability and statistics prerequisite track](https://github.com/jjames/llm-wiki/tree/main/lessons/prerequisites/03-probability-statistics).

**Recommended order:** prerequisite lessons → this notebook → Notebook 15 mastery  
**Time:** 100–130 minutes  
**Dependencies:** NumPy and Matplotlib only

### Learning goals

You will compute expectations from distributions, reason from joint tables, distinguish covariance from correlation, observe Monte Carlo error and the CLT, build confidence intervals, study power, and compare likelihood with a Beta posterior.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(20)


## 1. Distributions, expectation, and variance

A probability distribution assigns mass or density; it does not predict one certain outcome. Expectation is a probability-weighted average across possible outcomes. Variance measures expected squared deviation from that mean.

For $X\sim\mathrm{Bernoulli}(p)$, $E[X]=p$ and $\mathrm{Var}(X)=p(1-p)$. A Binomial count is a sum of independent Bernoulli variables.


In [ ]:
outcomes = np.arange(6)
pmf = np.array([0.05, 0.10, 0.20, 0.30, 0.25, 0.10])
expectation = np.sum(outcomes * pmf)
variance = np.sum((outcomes - expectation) ** 2 * pmf)
assert np.isclose(pmf.sum(), 1.0)

p = 0.3
samples = rng.binomial(1, p, size=100_000)
assert abs(samples.mean() - p) < 0.01
assert abs(samples.var() - p * (1 - p)) < 0.01

binomial_counts = rng.binomial(20, p, size=30_000)
assert abs(binomial_counts.mean() - 20 * p) < 0.1
assert abs(binomial_counts.var() - 20 * p * (1 - p)) < 0.2

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.arange(-0.5, 20.6, 1)
ax.hist(binomial_counts, bins=bins, density=True, color="C0", alpha=0.8)
ax.axvline(20 * p, color="C1", linestyle="--", label="expected count")
ax.set(xlabel="successes in 20 trials", ylabel="probability", title="A distribution describes repeated outcomes")
ax.legend(); ax.grid(alpha=0.2)
plt.show()
print(f"custom distribution: E[X]={expectation:.3f}, Var(X)={variance:.3f}")


## 2. Joint, marginal, conditional, and Bayes

A joint table $P(A,B)$ contains the full relationship between two discrete variables. Summing an axis gives a marginal. Dividing a joint slice by its marginal gives a conditional.

Bayes' rule changes which condition is treated as given:

$$P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}.$$

The denominator carries the base rate, which is why a high-sensitivity test can still have a modest positive predictive value for a rare condition.


In [ ]:
prevalence = 0.01
sensitivity = 0.95
false_positive_rate = 0.05
joint = np.array([
    [prevalence * sensitivity, prevalence * (1 - sensitivity)],
    [(1 - prevalence) * false_positive_rate, (1 - prevalence) * (1 - false_positive_rate)],
])
# Rows: condition present/absent. Columns: test positive/negative.
condition_marginal = joint.sum(axis=1)
test_marginal = joint.sum(axis=0)
positive_predictive_value = joint[0, 0] / test_marginal[0]
bayes_value = sensitivity * prevalence / test_marginal[0]

np.testing.assert_allclose(joint.sum(), 1.0)
np.testing.assert_allclose(condition_marginal, [prevalence, 1 - prevalence])
np.testing.assert_allclose(positive_predictive_value, bayes_value)
assert positive_predictive_value < 0.2

print("joint table (condition rows × test columns):")
print(joint)
print(f"P(condition | positive)={positive_predictive_value:.3f}, despite sensitivity={sensitivity:.2f}")


## 3. Covariance, correlation, and linear transformations

Covariance records whether variables deviate from their means together. Correlation standardizes covariance to $[-1,1]$, removing units. Neither measure alone establishes causality.

For a random vector $X$ and matrix $A$,

$$\operatorname{Cov}(AX)=A\operatorname{Cov}(X)A^T.$$


In [ ]:
true_covariance = np.array([[4.0, 1.8], [1.8, 1.5]])
samples = rng.multivariate_normal([2.0, -1.0], true_covariance, size=20_000)
sample_covariance = np.cov(samples, rowvar=False)
sample_correlation = np.corrcoef(samples, rowvar=False)
eigenvalues = np.linalg.eigvalsh(sample_covariance)
assert np.all(eigenvalues >= -1e-12)
np.testing.assert_allclose(np.diag(sample_correlation), np.ones(2))
np.testing.assert_allclose(sample_covariance, true_covariance, atol=0.08)

transform = np.array([[1.0, 1.0], [2.0, -0.5]])
transformed = samples @ transform.T
empirical_transformed_cov = np.cov(transformed, rowvar=False)
predicted_transformed_cov = transform @ sample_covariance @ transform.T
np.testing.assert_allclose(empirical_transformed_cov, predicted_transformed_cov, atol=1e-10)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(samples[:1500, 0], samples[:1500, 1], s=7, alpha=0.25)
ax.set(xlabel="X₁", ylabel="X₂", title=f"Covariance creates an oriented cloud (r={sample_correlation[0,1]:.2f})")
ax.grid(alpha=0.2)
plt.show()
print("sample covariance:")
print(sample_covariance)


## 4. Monte Carlo error and the law of large numbers

Monte Carlo estimates an expectation by a sample mean. Its standard error usually shrinks as $1/\sqrt{n}$—so obtaining one extra decimal digit can require about 100 times as many samples.


In [ ]:
# Estimate E[exp(-Z^2)] for Z ~ N(0,1); the exact value is 1/sqrt(3).
exact = 1 / np.sqrt(3)
sample_sizes = np.array([25, 100, 400, 1600, 6400])
replicates = 400
rmse = []
for n in sample_sizes:
    estimates = np.exp(-rng.normal(size=(replicates, n)) ** 2).mean(axis=1)
    rmse.append(np.sqrt(np.mean((estimates - exact) ** 2)))
rmse = np.array(rmse)
slope = np.polyfit(np.log(sample_sizes), np.log(rmse), 1)[0]
assert -0.65 < slope < -0.35
assert rmse[-1] < rmse[0] / 10

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(sample_sizes, rmse, marker="o", label="empirical RMSE")
ax.loglog(sample_sizes, rmse[0] * np.sqrt(sample_sizes[0] / sample_sizes), linestyle="--", label="1/√n reference")
ax.set(xlabel="sample size", ylabel="RMSE", title="Monte Carlo error shrinks slowly")
ax.legend(); ax.grid(True, which="both", alpha=0.25)
plt.show()
print(f"log-log slope={slope:.3f} (ideal -0.5)")


## 5. Sampling distributions, standard errors, and intervals

A sampling distribution describes how an estimator varies across hypothetical repeated samples. For the sample mean, the central limit theorem gives approximate normality under broad conditions:

$$\frac{\bar X-\mu}{\sigma/\sqrt n}\Rightarrow N(0,1).$$

A 95% confidence procedure is calibrated if it covers the fixed true parameter in about 95% of repeated samples. It does not mean a computed frequentist interval assigns 95% probability to that fixed parameter.


In [ ]:
population_mean = 2.0
population_sd = 3.0
sample_size = 40
repeated_samples = rng.normal(population_mean, population_sd, size=(20_000, sample_size))
sample_means = repeated_samples.mean(axis=1)
standardized = (sample_means - population_mean) / (population_sd / np.sqrt(sample_size))
lower = sample_means - 1.96 * population_sd / np.sqrt(sample_size)
upper = sample_means + 1.96 * population_sd / np.sqrt(sample_size)
coverage = np.mean((lower <= population_mean) & (population_mean <= upper))

assert abs(standardized.mean()) < 0.03
assert abs(standardized.std() - 1.0) < 0.03
assert 0.94 < coverage < 0.96

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(standardized, bins=60, density=True, alpha=0.75, color="C0")
grid = np.linspace(-4, 4, 300)
ax.plot(grid, np.exp(-grid**2 / 2) / np.sqrt(2 * np.pi), color="C1", label="standard normal")
ax.set(xlabel="standardized sample mean", ylabel="density", title="Sampling distribution of the mean")
ax.legend(); ax.grid(alpha=0.2)
plt.show()
print(f"95% interval coverage across repeated samples: {coverage:.3f}")


## 6. Likelihood, MLE, and a Beta posterior

The likelihood treats observed data as fixed and varies the parameter. For $k$ successes in $n$ Bernoulli trials, the MLE is $k/n$. With a $\mathrm{Beta}(\alpha,\beta)$ prior, the posterior is

$$p\mid k,n\sim\mathrm{Beta}(\alpha+k,\beta+n-k).$$

The posterior mean compromises between the prior mean and sample proportion, with strength determined by their effective sample sizes.


In [ ]:
successes, trials = 7, 10
mle = successes / trials
alpha_prior, beta_prior = 2.0, 5.0
alpha_post = alpha_prior + successes
beta_post = beta_prior + trials - successes
prior_mean = alpha_prior / (alpha_prior + beta_prior)
posterior_mean = alpha_post / (alpha_post + beta_post)
posterior_draws = rng.beta(alpha_post, beta_post, size=200_000)
credible_interval = np.quantile(posterior_draws, [0.025, 0.975])

assert prior_mean < posterior_mean < mle
assert credible_interval[0] < posterior_mean < credible_interval[1]

parameter_grid = np.linspace(0.001, 0.999, 500)
log_likelihood = successes * np.log(parameter_grid) + (trials - successes) * np.log(1 - parameter_grid)
likelihood = np.exp(log_likelihood - log_likelihood.max())
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(parameter_grid, likelihood, label="scaled likelihood")
ax.hist(posterior_draws, bins=80, density=True, alpha=0.35, label="posterior density")
ax.axvline(mle, color="C2", linestyle="--", label="MLE")
ax.axvline(posterior_mean, color="C3", linestyle="--", label="posterior mean")
ax.set(xlabel="Bernoulli probability p", title="Likelihood and posterior answer different questions")
ax.legend(); ax.grid(alpha=0.2)
plt.show()
print(f"prior mean={prior_mean:.3f}, MLE={mle:.3f}, posterior mean={posterior_mean:.3f}")
print("95% posterior credible interval:", credible_interval)


## 7. Tests, Type I error, and power

A test statistic measures incompatibility with a null model. A p-value is the probability—assuming the null and the analysis plan—of a result at least as extreme as observed. It is not the probability that the null is true.

The significance level controls long-run Type I error. Power is the probability of rejecting the null under a particular alternative; it rises with effect size and sample size.


In [ ]:
def rejection_rate(effect, n, repetitions=20_000, alpha=0.05):
    # Known unit variance: z-test for a one-sample mean.
    sample_means = rng.normal(effect, 1 / np.sqrt(n), size=repetitions)
    z = sample_means * np.sqrt(n)
    return np.mean(np.abs(z) > 1.959963984540054)

null_rate = rejection_rate(effect=0.0, n=50)
low_power = rejection_rate(effect=0.25, n=20)
high_power = rejection_rate(effect=0.25, n=200)
assert 0.04 < null_rate < 0.06
assert high_power > low_power
assert high_power > 0.9

print(f"Type I error under null: {null_rate:.3f}")
print(f"power for effect 0.25: n=20 -> {low_power:.3f}; n=200 -> {high_power:.3f}")


## Cumulative mastery check

1. Distinguish a realized sample, a population distribution, and a sampling distribution.
2. Recompute the diagnostic-test posterior odds from prior odds and a likelihood ratio.
3. Give two data-generating stories with equal correlation but different causal structure.
4. Why does four times the Monte Carlo sample size roughly halve standard error?
5. Contrast a 95% confidence interval with a 95% credible interval.
6. What changes power without changing the chosen Type I error rate?

**Next:** Notebook 15 applies these ideas to paired model evaluation, bootstrap intervals, randomization tests, subgroup shift, and Bayesian updating.
